In [6]:
!pip install transformers torch sentencepiece PyPDF2


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import torch
from transformers import PegasusTokenizer, PegasusForConditionalGeneration
from PyPDF2 import PdfReader
import re

In [8]:
import sys
print(sys.executable)

C:\Users\tina\practice\data-cleaning\netflixshows\venv\Scripts\python.exe


In [9]:
import sys
!{sys.executable} -m pip install torch transformers sentencepiece PyPDF2


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\tina\practice\data-cleaning\netflixshows\venv\Scripts\python.exe -m pip install --upgrade pip


In [10]:
import torch
from transformers import PegasusTokenizer, PegasusForConditionalGeneration
from PyPDF2 import PdfReader
import re

print("Torch version:", torch.__version__) #sanity check4me

Torch version: 2.10.0+cpu


In [11]:
def extract_text_from_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:  # avoid None
            text += page_text + "\n"

    return text

In [12]:
pdf_path = "datasets/pdf.pdf"  # make sure file exists
raw_text = extract_text_from_pdf(pdf_path)

print(raw_text[:1000])

Jack and the Beanstalk
Flora Annie Steel
A long long time ago, when most of the world was young and folk did what they liked because all 
things were good, there lived a boy called Jack.
His father was bed-ridden, and his mother, a good soul, was busy early morns and late eyes 
planning and placing how to support her sick husband and her young son by selling the milk 
and butter which Milky-White, the beautiful cow, gave them without stint. For it was summer-
time. But winter came on; the herbs of the fields took refuge from the frosts in the warm earth, 
and though his mother sent Jack to gather what fodder he could get in the hedgerows, he came 
back as often as not with a very empty sack; for Jack’s eyes were so often full of wonder at all the 
things he saw that sometimes he forgot to work!
So it came to pass that one morning Milky-White gave no milk at all—not one drain! Then the 
good hard-working mother threw her apron over her head and sobbed:
“What shall we do? What shall we d

In [13]:
def clean_text(text):
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9.,;:()\- ]', '', text)
    return text.strip()

In [14]:
cleaned_text = clean_text(raw_text)
print(cleaned_text[:1000])

Jack and the Beanstalk Flora Annie Steel A long long time ago, when most of the world was young and folk did what they liked because all things were good, there lived a boy called Jack. His father was bed-ridden, and his mother, a good soul, was busy early morns and late eyes planning and placing how to support her sick husband and her young son by selling the milk and butter which Milky-White, the beautiful cow, gave them without stint. For it was summer- time. But winter came on; the herbs of the fields took refuge from the frosts in the warm earth, and though his mother sent Jack to gather what fodder he could get in the hedgerows, he came back as often as not with a very empty sack; for Jacks eyes were so often full of wonder at all the things he saw that sometimes he forgot to work So it came to pass that one morning Milky-White gave no milk at allnot one drain Then the good hard-working mother threw her apron over her head and sobbed: What shall we do What shall we do Now Jack lo

In [15]:
model_name = "google/pegasus-xsum"

tokenizer = PegasusTokenizer.from_pretrained(model_name)
model = PegasusForConditionalGeneration.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

C:\Users\tina\practice\data-cleaning\netflixshows\venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tina\.cache\huggingface\hub\models--google--pegasus-xsum. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-xsum
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/259 [00:00<?, ?B/s]

In [16]:
def chunk_text(text, chunk_size=600):  # smaller = safer
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks

In [17]:
chunks = chunk_text(cleaned_text)
print("Total chunks:", len(chunks))

Total chunks: 7


In [25]:
def summarize_chunk(chunk):
    # Ensure no more than 1024 tokens
    inputs = tokenizer(
        chunk,
        return_tensors="pt",
        truncation=True,
        max_length=1024,  # Pegasus limit
        padding="longest"
    )

    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=150,
            min_length=40,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True
        )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [26]:
def chunk_text(text, chunk_size=400):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunks.append(" ".join(words[i:i+chunk_size]))
    return chunks

chunks = chunk_text(cleaned_text)
print("Total chunks:", len(chunks))

Total chunks: 10


In [27]:
summaries = []

for i, chunk in enumerate(chunks, start=1):
    print(f"Summarizing chunk {i}/{len(chunks)}...")
    summary = summarize_chunk(chunk)
    summaries.append(summary)

# Combine all chunk summaries into a final summary
final_summary = " ".join(summaries)

print("\nFINAL SUMMARY:\n")
print(final_summary)

Summarizing chunk 1/10...
Summarizing chunk 2/10...
Summarizing chunk 3/10...
Summarizing chunk 4/10...
Summarizing chunk 5/10...
Summarizing chunk 6/10...
Summarizing chunk 7/10...
Summarizing chunk 8/10...
Summarizing chunk 9/10...
Summarizing chunk 10/10...

FINAL SUMMARY:

Jack and the Beanstalk is a children's book by Annie Steel, published by Little, Brown Books for Young Readers, and available on Amazon, Barnes & Noble, and other digital outlets., Jack was walking along the road when he saw a queer little old man on the road who called out, Good-morning, Jack Good-morning, replied Jack, with a polite bow, wondering how the queer little old man happened to know his name; though, to be sure, Jacks were as plentiful as blackberries. Jack was nine years old, and he was walking home from school when he came across a little queer little old man, who was sitting on a bench, and he said to him: What the queer little old man said was true. Jack, having had no supper, was hungry as a hunt